In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_2305349;

CREATE VOLUME IF NOT EXISTS workspace.ecommerce_2305349.raw;


In [0]:
MY_ID = "2305349"

VOL = f"/Volumes/workspace/ecommerce_{MY_ID}/raw"

print("Project ID:", MY_ID)
print("Raw volume:", VOL)

Project ID: 2305349
Raw volume: /Volumes/workspace/ecommerce_2305349/raw


In [0]:
from pyspark.sql import functions as F

# Reproducibility
SEED = 2305349

print("SEED =", SEED)
print("Raw volume =", VOL)


SEED = 2305349
Raw volume = /Volumes/workspace/ecommerce_2305349/raw


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.ecommerce_2305349;

CREATE VOLUME IF NOT EXISTS workspace.ecommerce_2305349.raw;

In [0]:
MY_ID = "2305349"

VOL = f"/Volumes/workspace/ecommerce_{MY_ID}/raw"

print("Project ID:", MY_ID)
print("Raw volume:", VOL)

Project ID: 2305349
Raw volume: /Volumes/workspace/ecommerce_2305349/raw


In [0]:
from pyspark.sql import functions as F

SEED = 2305349

print("SEED =", SEED)
print("Raw volume =", VOL)

SEED = 2305349
Raw volume = /Volumes/workspace/ecommerce_2305349/raw


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

In [0]:
#customers
SEED = 2305349
customers = (
    spark.range(2000)
    .select(
        F.format_string("C%05d", F.col("id")).alias("customer_id"),
        F.concat(F.lit("Customer_"), F.col("id")).alias("customer_name"),
        F.concat(F.lit("city_"), (F.col("id") % 20)).alias("city"),
        F.concat(F.lit("state_"), (F.col("id") % 10)).alias("state")
    )
)

In [0]:
#sellers
sellers = (
    spark.range(100)
    .select(
        F.format_string("S%03d", F.col("id")).alias("seller_id"),
        F.concat(F.lit("Seller_"), F.col("id")).alias("seller_name"),
        F.concat(F.lit("state_"), (F.col("id") % 10)).alias("state")
    )
)

In [0]:
#products
categories = [
    "Electronics",
    "Fashion",
    "Home & Kitchen",
    "Beauty",
    "Sports",
    "Books",
    "Toys",
    "Grocery",
    "Furniture",
    "Automotive"
]

products = (
    spark.range(300)
    .select(
        F.format_string("P%04d", F.col("id")).alias("product_id"),
        F.concat(F.lit("Product_"), F.col("id")).alias("product_name"),
        F.when((F.col("id") % 10) == 0, "Electronics")
         .when((F.col("id") % 10) == 1, "Fashion")
         .when((F.col("id") % 10) == 2, "Home & Kitchen")
         .when((F.col("id") % 10) == 3, "Beauty")
         .when((F.col("id") % 10) == 4, "Sports")
         .when((F.col("id") % 10) == 5, "Books")
         .when((F.col("id") % 10) == 6, "Toys")
         .when((F.col("id") % 10) == 7, "Grocery")
         .when((F.col("id") % 10) == 8, "Furniture")
         .otherwise("Automotive")
         .alias("category"),
        F.round(
            20 + F.pmod(
                F.hash(F.col("id"), F.lit(SEED)),
                F.lit(981)
            ) / 10,
            2
        ).alias("base_price")
    )
)

print("Products :", products.count())

Products : 300


In [0]:
#ORDERS

orders_base = (
    spark.range(10000)
    .select(
        F.col("id"),
        F.format_string("O%06d", F.col("id")).alias("order_id"),
        F.format_string(
            "C%05d",
            F.pmod(F.hash(F.col("id"), F.lit(11)), F.lit(2000))
        ).alias("customer_id"),
        F.format_string(
            "S%03d",
            (F.col("id") % 100) + 1
        ).alias("seller_id"),
        F.date_add(
            F.lit("2025-01-01"),
            F.pmod(
                F.hash(F.col("id"), F.lit(12)),
                F.lit(365)
            )
        ).alias("order_date")
    )
)

orders = (
    orders_base
    .withColumn(
        "delivery_days",
        F.when(
            F.pmod(F.hash(F.col("id"), F.lit(20)), F.lit(10)) < 7,
            3 + F.pmod(F.hash(F.col("id"), F.lit(21)), F.lit(3))
        ).otherwise(
            8 + F.pmod(F.hash(F.col("id"), F.lit(22)), F.lit(5))
        )
    )
    .withColumn(
        "estimated_delivery_date",
        F.date_add(F.col("order_date"), 7)
    )
    .withColumn(
        "delivered_date",
        F.date_add(
            F.col("order_date"),
            F.col("delivery_days")
        )
    )
    .withColumn(
        "order_status",
        F.when(
            F.pmod(F.hash(F.col("id"), F.lit(23)), F.lit(20)) == 0,
            "Cancelled"
        ).otherwise("Delivered")
    )
    .drop("id")
)

In [0]:
#ORDER ITEMS

items_base = (
    spark.range(10000)
    .select(
        F.col("id"),
        F.format_string("O%06d", F.col("id")).alias("order_id")
    )
)

# 1–3 items per order
order_items = (
    items_base
    .withColumn(
        "item_count",
        1 + F.pmod(
            F.hash(F.col("id"), F.lit(30)),
            F.lit(3)
        )
    )
    .withColumn(
        "item_no",
        F.explode(
            F.sequence(
                F.lit(1),
                F.col("item_count")
            )
        )
    )
    .withColumn(
        "product_id",
        F.format_string(
            "P%04d",
            F.pmod(
                F.hash(
                    F.col("id"),
                    F.col("item_no"),
                    F.lit(31)
                ),
                F.lit(300)
            )
        )
    )
    .withColumn(
        "quantity",
        1 + F.pmod(
            F.hash(
                F.col("id"),
                F.col("item_no"),
                F.lit(32)
            ),
            F.lit(3)
        )
    )
    .select(
        "order_id",
        "item_no",
        "product_id",
        "quantity"
    )
)

# Add product prices
order_items = (
    order_items
    .join(
        products.select("product_id", "base_price"),
        "product_id",
        "left"
    )
    .withColumn(
        "unit_price",
        F.round(
            F.col("base_price") *
            (
                F.lit(0.9) +
                F.pmod(
                    F.hash(
                        F.col("order_id"),
                        F.col("item_no"),
                        F.lit(33)
                    ),
                    F.lit(21)
                ) / 100
            ),
            2
        )
    )
    .drop("base_price")
)


In [0]:
#PAYMENTS
payments = (
    orders
    .select(
        "order_id"
    )
    .withColumn(
        "payment_method",
        F.element_at(
            F.array(
                F.lit("Credit Card"),
                F.lit("Debit Card"),
                F.lit("UPI"),
                F.lit("Net Banking")
            ),
            F.pmod(
                F.hash(F.col("order_id"), F.lit(40)),
                F.lit(4)
            ) + 1
        )
    )
    .join(
        order_items
        .groupBy("order_id")
        .agg(
            F.round(
                F.sum(
                    F.col("quantity") * F.col("unit_price")
                ),
                2
            ).alias("payment_amount")
        ),
        "order_id",
        "left"
    )
)


In [0]:
reviews = (
    orders
    .select(
        "order_id",
        "delivered_date",
        "estimated_delivery_date"
    )
    .withColumn(
        "review_score",
        F.when(
            F.col("delivered_date") > F.col("estimated_delivery_date"),
            2 + F.pmod(
                F.hash(F.col("order_id"), F.lit(50)),
                F.lit(3)
            )
        ).otherwise(
            3 + F.pmod(
                F.hash(F.col("order_id"), F.lit(51)),
                F.lit(3)
            )
        )
    )
    .withColumn(
        "review_date",
        F.date_add(F.col("delivered_date"), 2)
    )
    .select(
        "order_id",
        "review_score",
        "review_date"
    )
)


In [0]:
#showing raw datasets
print("Customers :", customers.count())
print("Sellers   :", sellers.count())
print("Products  :", products.count())
print("Orders    :", orders.count())
print("Items     :", order_items.count())
print("Payments  :", payments.count())
print("Reviews   :", reviews.count())

Customers : 2000
Sellers   : 100
Products  : 300
Orders    : 10000
Items     : 20003
Payments  : 10000
Reviews   : 10000


In [0]:
#Bronze Layer Rawdata
VOL = "/Volumes/workspace/ecommerce_2305349/raw"

datasets = {
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews
}

for name, df in datasets.items():
    path = f"{VOL}/{name}"

    (
        df.write
        .mode("overwrite")
        .option("header", True)
        .csv(path)
    )

    print(f"Written: {name} -> {path}")

Written: customers -> /Volumes/workspace/ecommerce_2305349/raw/customers
Written: sellers -> /Volumes/workspace/ecommerce_2305349/raw/sellers
Written: products -> /Volumes/workspace/ecommerce_2305349/raw/products
Written: orders -> /Volumes/workspace/ecommerce_2305349/raw/orders
Written: order_items -> /Volumes/workspace/ecommerce_2305349/raw/order_items
Written: payments -> /Volumes/workspace/ecommerce_2305349/raw/payments
Written: reviews -> /Volumes/workspace/ecommerce_2305349/raw/reviews


In [0]:
#verify bronze
bronze_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce_2305349/raw/orders")
)

print("Bronze orders rows:", bronze_orders.count())
print("Bronze orders columns:", len(bronze_orders.columns))

display(bronze_orders.limit(5))

Bronze orders rows: 10000
Bronze orders columns: 8


order_id,customer_id,seller_id,order_date,delivery_days,estimated_delivery_date,delivered_date,order_status
O000000,C00646,S001,2025-01-13,3,2025-01-20,2025-01-16,Delivered
O000001,C00656,S002,2025-07-25,3,2025-08-01,2025-07-28,Delivered
O000002,C01992,S003,2025-07-29,4,2025-08-05,2025-08-02,Delivered
O000003,C01988,S004,2025-04-20,4,2025-04-27,2025-04-24,Delivered
O000004,C01104,S005,2025-12-20,5,2025-12-27,2025-12-25,Delivered


In [0]:
#Silver- brnze data quality
orders_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/ecommerce_2305349/raw/orders")
)

print("Total rows:", orders_bronze.count())

print("\nColumns:")
orders_bronze.printSchema()

print("\nNull counts:")
orders_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in orders_bronze.columns
]).show()

print("\nDuplicate order IDs:")
print(
    orders_bronze
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nOrder status:")
orders_bronze.groupBy("order_status").count().show()

print("\nSample:")
display(orders_bronze.limit(10))

Total rows: 10000

Columns:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- estimated_delivery_date: date (nullable = true)
 |-- delivered_date: date (nullable = true)
 |-- order_status: string (nullable = true)


Null counts:
+--------+-----------+---------+----------+-------------+-----------------------+--------------+------------+
|order_id|customer_id|seller_id|order_date|delivery_days|estimated_delivery_date|delivered_date|order_status|
+--------+-----------+---------+----------+-------------+-----------------------+--------------+------------+
|       0|          0|        0|         0|            0|                      0|             0|           0|
+--------+-----------+---------+----------+-------------+-----------------------+--------------+------------+


Duplicate order IDs:
0

Order status:
+----------

order_id,customer_id,seller_id,order_date,delivery_days,estimated_delivery_date,delivered_date,order_status
O000000,C00646,S001,2025-01-13,3,2025-01-20,2025-01-16,Delivered
O000001,C00656,S002,2025-07-25,3,2025-08-01,2025-07-28,Delivered
O000002,C01992,S003,2025-07-29,4,2025-08-05,2025-08-02,Delivered
O000003,C01988,S004,2025-04-20,4,2025-04-27,2025-04-24,Delivered
O000004,C01104,S005,2025-12-20,5,2025-12-27,2025-12-25,Delivered
O000005,C01384,S006,2025-10-29,9,2025-11-05,2025-11-07,Delivered
O000006,C01236,S007,2025-01-15,4,2025-01-22,2025-01-19,Delivered
O000007,C00916,S008,2025-05-08,3,2025-05-15,2025-05-11,Delivered
O000008,C01429,S009,2025-12-03,8,2025-12-10,2025-12-11,Delivered
O000009,C01188,S010,2025-06-29,4,2025-07-06,2025-07-03,Delivered


In [0]:
# SILVER LAYER - CLEAN AND INTEGRATE E-COMMERCE DATA
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "/Volumes/workspace/ecommerce_2305349/raw"

In [0]:
# 1. READ BRONZE DATA
orders_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/orders")
)

customers_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/customers")
)

sellers_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/sellers")
)

products_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/products")
)

items_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/order_items")
)

reviews_b = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/reviews")
)


In [0]:
# 2. CLEAN / STANDARDIZE ORDERS
orders_s = (
    orders_b
    .dropDuplicates(["order_id"])
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn(
        "estimated_delivery_date",
        F.to_date("estimated_delivery_date")
    )
    .withColumn(
        "delivered_date",
        F.to_date("delivered_date")
    )
    .withColumn(
        "delivery_days",
        F.col("delivery_days").cast("int")
    )
)


In [0]:
items_s = (
    items_b
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
)

products_s = (
    products_b
    .withColumn("base_price", F.col("base_price").cast("double"))
)

items_enriched = (
    items_s
    .join(
        products_s.select(
            "product_id",
            "product_name",
            "category"
        ),
        on="product_id",
        how="left"
    )
    .withColumn(
        "line_value",
        F.round(
            F.col("quantity") * F.col("unit_price"),
            2
        )
    )
)

In [0]:
# 4. AGGREGATE ITEMS TO ORDER LEVEL
order_items_summary = (
    items_enriched
    .groupBy("order_id")
    .agg(
        F.sum("quantity").alias("item_count"),
        F.round(
            F.sum("line_value"),
            2
        ).alias("order_value"),
        F.countDistinct("product_id").alias(
            "unique_products"
        ),
        F.concat_ws(
            ", ",
            F.sort_array(
                F.collect_set("category")
            )
        ).alias("categories")
    )
)


In [0]:
# 5. ADD CUSTOMER INFORMATION
customers_s = customers_b.select(
    "customer_id",
    "customer_name",
    "city",
    "state"
)

orders_customer = (
    orders_s
    .join(
        customers_s,
        on="customer_id",
        how="left"
    )
)

In [0]:
# 6. ADD SELLER INFORMATION
sellers_s = sellers_b.select(
    "seller_id",
    "seller_name",
    F.col("state").alias("seller_state")
)

orders_customer_seller = (
    orders_customer
    .join(
        sellers_s,
        on="seller_id",
        how="left"
    )
)


In [0]:
# 7. ADD ORDER VALUE / PRODUCT INFORMATION
orders_items = (
    orders_customer_seller
    .join(
        order_items_summary,
        on="order_id",
        how="left"
    )
)

In [0]:
# 8. ADD REVIEW INFORMATION
reviews_s = (
    reviews_b
    .withColumn(
        "review_score",
        F.col("review_score").cast("int")
    )
    .withColumn(
        "review_date",
        F.to_date("review_date")
    )
)

silver_orders = (
    orders_items
    .join(
        reviews_s,
        on="order_id",
        how="left"
    )
)

In [0]:
# 9. DERIVE DELIVERY STATUS
silver_orders = (
    silver_orders
    .withColumn(
        "delivery_status",
        F.when(
            F.col("order_status") == "Cancelled",
            "Cancelled"
        )
        .when(
            F.col("delivered_date") >
            F.col("estimated_delivery_date"),
            "Late"
        )
        .when(
            F.col("delivered_date") <=
            F.col("estimated_delivery_date"),
            "On Time"
        )
        .otherwise("Unknown")
    )
    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("delivery_status") == "Late",
            F.datediff(
                F.col("delivered_date"),
                F.col("estimated_delivery_date")
            )
        ).otherwise(0)
    )
)


In [0]:
# 10. SILVER QUALITY CHECKS
print("Silver rows:", silver_orders.count())
print("Silver columns:", len(silver_orders.columns))

print("\nDelivery status:")
silver_orders.groupBy(
    "delivery_status"
).count().orderBy(
    F.desc("count")
).show()

print("\nNull order IDs:")
print(
    silver_orders
    .filter(F.col("order_id").isNull())
    .count()
)

print("\nDuplicate order IDs:")
print(
    silver_orders
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nSilver sample:")
display(silver_orders.limit(10))

Silver rows: 10000
Silver columns: 21

Delivery status:
+---------------+-----+
|delivery_status|count|
+---------------+-----+
|        On Time| 6607|
|           Late| 2896|
|      Cancelled|  497|
+---------------+-----+


Null order IDs:
0

Duplicate order IDs:
0

Silver sample:


order_id,seller_id,customer_id,order_date,delivery_days,estimated_delivery_date,delivered_date,order_status,customer_name,city,state,seller_name,seller_state,item_count,order_value,unique_products,categories,review_score,review_date,delivery_status,delivery_delay_days
O000035,S036,C00847,2025-10-31,4,2025-11-07,2025-11-04,Delivered,Customer_847,city_7,state_7,Seller_36,state_6,2,70.04,1,Automotive,3,2025-11-06,On Time,0
O000037,S038,C00623,2025-03-06,11,2025-03-13,2025-03-17,Delivered,Customer_623,city_3,state_3,Seller_38,state_8,2,225.5,1,Books,2,2025-03-19,Late,4
O000062,S063,C01301,2025-06-07,3,2025-06-14,2025-06-10,Delivered,Customer_1301,city_1,state_1,Seller_63,state_3,5,357.18,2,"Automotive, Toys",4,2025-06-12,On Time,0
O000078,S079,C01515,2025-11-11,3,2025-11-18,2025-11-14,Delivered,Customer_1515,city_15,state_5,Seller_79,state_9,6,530.0,3,"Furniture, Grocery, Toys",5,2025-11-16,On Time,0
O000085,S086,C00009,2025-11-02,4,2025-11-09,2025-11-06,Delivered,Customer_9,city_9,state_9,Seller_86,state_6,6,492.42,3,"Beauty, Grocery",4,2025-11-08,On Time,0
O000090,S091,C00272,2025-04-09,5,2025-04-16,2025-04-14,Delivered,Customer_272,city_12,state_2,Seller_91,state_1,5,401.96,3,"Beauty, Furniture",4,2025-04-16,On Time,0
O000098,S099,C01790,2025-04-22,4,2025-04-29,2025-04-26,Delivered,Customer_1790,city_10,state_0,Seller_99,state_9,3,289.59,1,Toys,3,2025-04-28,On Time,0
O000099,S100,C01863,2025-02-08,5,2025-02-15,2025-02-13,Delivered,Customer_1863,city_3,state_3,null,null,5,309.98,2,Toys,5,2025-02-15,On Time,0
O000114,S015,C01492,2025-03-26,5,2025-04-02,2025-03-31,Delivered,Customer_1492,city_12,state_2,Seller_15,state_5,3,133.23,1,Toys,3,2025-04-02,On Time,0
O000135,S036,C00243,2025-01-27,3,2025-02-03,2025-01-30,Delivered,Customer_243,city_3,state_3,Seller_36,state_6,1,76.76,1,Home & Kitchen,4,2025-02-01,On Time,0


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.ecommerce_2305349.silver;

In [0]:
# SAVE SILVER LAYER
SILVER_PATH = "/Volumes/workspace/ecommerce_2305349/silver/orders"

(
    silver_orders.write
    .mode("overwrite")
    .format("delta")
    .save(SILVER_PATH)
)

print("Silver table written to:")
print(SILVER_PATH)

print("Silver row count:", spark.read.format("delta").load(SILVER_PATH).count())

Silver table written to:
/Volumes/workspace/ecommerce_2305349/silver/orders
Silver row count: 10000


In [0]:
# GOLD LAYER - LOAD SILVER DATA
SILVER_PATH = "/Volumes/workspace/ecommerce_2305349/silver/orders"

silver = (
    spark.read
    .format("delta")
    .load(SILVER_PATH)
)

print("Silver rows:", silver.count())
print("Silver columns:", len(silver.columns))

display(silver.limit(5))

Silver rows: 10000
Silver columns: 21


order_id,seller_id,customer_id,order_date,delivery_days,estimated_delivery_date,delivered_date,order_status,customer_name,city,state,seller_name,seller_state,item_count,order_value,unique_products,categories,review_score,review_date,delivery_status,delivery_delay_days
O000035,S036,C00847,2025-10-31,4,2025-11-07,2025-11-04,Delivered,Customer_847,city_7,state_7,Seller_36,state_6,2,70.04,1,Automotive,3,2025-11-06,On Time,0
O000037,S038,C00623,2025-03-06,11,2025-03-13,2025-03-17,Delivered,Customer_623,city_3,state_3,Seller_38,state_8,2,225.5,1,Books,2,2025-03-19,Late,4
O000062,S063,C01301,2025-06-07,3,2025-06-14,2025-06-10,Delivered,Customer_1301,city_1,state_1,Seller_63,state_3,5,357.18,2,"Automotive, Toys",4,2025-06-12,On Time,0
O000078,S079,C01515,2025-11-11,3,2025-11-18,2025-11-14,Delivered,Customer_1515,city_15,state_5,Seller_79,state_9,6,530.0,3,"Furniture, Grocery, Toys",5,2025-11-16,On Time,0
O000085,S086,C00009,2025-11-02,4,2025-11-09,2025-11-06,Delivered,Customer_9,city_9,state_9,Seller_86,state_6,6,492.42,3,"Beauty, Grocery",4,2025-11-08,On Time,0


In [0]:
# GOLD 1 - EXECUTIVE DELIVERY KPIs
gold_kpis = silver.agg(
    F.count("*").alias("total_orders"),

    F.sum(
        F.when(F.col("order_status") == "Delivered", 1)
        .otherwise(0)
    ).alias("delivered_orders"),

    F.sum(
        F.when(F.col("order_status") == "Cancelled", 1)
        .otherwise(0)
    ).alias("cancelled_orders"),

    F.sum(
        F.when(F.col("delivery_status") == "On Time", 1)
        .otherwise(0)
    ).alias("on_time_orders"),

    F.sum(
        F.when(F.col("delivery_status") == "Late", 1)
        .otherwise(0)
    ).alias("late_orders"),

    F.round(
        F.avg(
            F.when(
                F.col("order_status") == "Delivered",
                F.col("delivery_days")
            )
        ),
        2
    ).alias("avg_delivery_days"),

    F.round(
        F.sum(
            F.when(
                F.col("order_status") == "Delivered",
                F.col("order_value")
            ).otherwise(0)
        ),
        2
    ).alias("total_revenue"),

    F.round(
        F.avg(
            F.when(
                F.col("order_status") == "Delivered",
                F.col("order_value")
            )
        ),
        2
    ).alias("avg_order_value")
).withColumn(
    "on_time_delivery_rate",
    F.round(
        F.col("on_time_orders") /
        F.col("delivered_orders") * 100,
        2
    )
).withColumn(
    "cancellation_rate",
    F.round(
        F.col("cancelled_orders") /
        F.col("total_orders") * 100,
        2
    )
)

display(gold_kpis)


total_orders,delivered_orders,cancelled_orders,on_time_orders,late_orders,avg_delivery_days,total_revenue,avg_order_value,on_time_delivery_rate,cancellation_rate
10000,9503,497,6607,2896,5.84,2649693.93,278.83,69.53,4.97


In [0]:
# GOLD 2 - MONTHLY DELIVERY PERFORMANCE
gold_monthly = (
    silver
    .withColumn(
        "order_month",
        F.date_format("order_date", "yyyy-MM")
    )
    .groupBy("order_month")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Delivered", 1
            ).otherwise(0)
        ).alias("delivered_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Cancelled", 1
            ).otherwise(0)
        ).alias("cancelled_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "On Time", 1
            ).otherwise(0)
        ).alias("on_time_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "Late", 1
            ).otherwise(0)
        ).alias("late_orders"),

        F.round(
            F.avg(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("delivery_days")
                )
            ),
            2
        ).alias("avg_delivery_days"),

        F.round(
            F.sum(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("order_value")
                ).otherwise(0)
            ),
            2
        ).alias("revenue")
    )
    .withColumn(
        "on_time_delivery_rate",
        F.round(
            F.col("on_time_orders") /
            F.col("delivered_orders") * 100,
            2
        )
    )
    .orderBy("order_month")
)

display(gold_monthly)

order_month,total_orders,delivered_orders,cancelled_orders,on_time_orders,late_orders,avg_delivery_days,revenue,on_time_delivery_rate
2025-01,846,803,43,546,257,5.91,226382.94,68.0
2025-02,792,759,33,520,239,5.92,215364.94,68.51
2025-03,846,806,40,576,230,5.71,226782.26,71.46
2025-04,808,785,23,553,232,5.8,223537.52,70.45
2025-05,825,779,46,535,244,5.9,208189.73,68.68
2025-06,793,752,41,515,237,5.83,212361.53,68.48
2025-07,846,799,47,555,244,5.9,216651.83,69.46
2025-08,891,848,43,600,248,5.76,231982.75,70.75
2025-09,876,828,48,578,250,5.86,229972.24,69.81
2025-10,841,802,39,570,232,5.78,226809.05,71.07


In [0]:
# GOLD 3 - SELLER DELIVERY PERFORMANCE
gold_sellers = (
    silver
    .groupBy("seller_id")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Delivered", 1
            ).otherwise(0)
        ).alias("delivered_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Cancelled", 1
            ).otherwise(0)
        ).alias("cancelled_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "On Time", 1
            ).otherwise(0)
        ).alias("on_time_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "Late", 1
            ).otherwise(0)
        ).alias("late_orders"),

        F.round(
            F.avg(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("delivery_days")
                )
            ),
            2
        ).alias("avg_delivery_days"),

        F.round(
            F.sum(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("order_value")
                ).otherwise(0)
            ),
            2
        ).alias("revenue")
    )
    .withColumn(
        "on_time_delivery_rate",
        F.round(
            F.col("on_time_orders") /
            F.col("delivered_orders") * 100,
            2
        )
    )
    .filter(F.col("total_orders") >= 50)
    .orderBy(F.desc("on_time_delivery_rate"))
)

display(gold_sellers)

seller_id,total_orders,delivered_orders,cancelled_orders,on_time_orders,late_orders,avg_delivery_days,revenue,on_time_delivery_rate
S053,100,91,9,76,15,4.8,24202.98,83.52
S061,100,93,7,74,19,5.32,25999.86,79.57
S062,100,95,5,75,20,5.41,26561.33,78.95
S068,100,91,9,71,20,5.13,25390.44,78.02
S039,100,95,5,74,21,5.42,25673.62,77.89
S032,100,93,7,72,21,5.11,25853.52,77.42
S040,100,95,5,72,23,5.54,24467.31,75.79
S080,100,95,5,72,23,5.58,28120.85,75.79
S001,100,99,1,75,24,5.47,24567.94,75.76
S048,100,94,6,71,23,5.64,24374.91,75.53


In [0]:
# GOLD 4 - LOAD PRODUCT DATA
PRODUCTS_PATH = "/Volumes/workspace/ecommerce_2305349/raw/products"

products_gold = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(PRODUCTS_PATH)
)

print("Product rows:", products_gold.count())
print("Product columns:", len(products_gold.columns))

print("Product schema:")
products_gold.printSchema()

display(products_gold.limit(10))

Product rows: 300
Product columns: 4
Product schema:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- base_price: double (nullable = true)



product_id,product_name,category,base_price
P0187,Product_187,Grocery,108.7
P0188,Product_188,Furniture,115.0
P0189,Product_189,Automotive,39.2
P0190,Product_190,Electronics,109.2
P0191,Product_191,Fashion,39.7
P0192,Product_192,Home & Kitchen,60.0
P0193,Product_193,Beauty,87.9
P0194,Product_194,Sports,40.3
P0195,Product_195,Books,98.7
P0196,Product_196,Toys,103.8


In [0]:
# GOLD 4 - CATEGORY PERFORMANCE
gold_category = (
    silver
    .groupBy("categories")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Delivered", 1
            ).otherwise(0)
        ).alias("delivered_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Cancelled", 1
            ).otherwise(0)
        ).alias("cancelled_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "On Time", 1
            ).otherwise(0)
        ).alias("on_time_orders"),

        F.sum(
            F.when(
                F.col("delivery_status") == "Late", 1
            ).otherwise(0)
        ).alias("late_orders"),

        F.round(
            F.avg(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("delivery_days")
                )
            ),
            2
        ).alias("avg_delivery_days"),

        F.round(
            F.sum(
                F.when(
                    F.col("order_status") == "Delivered",
                    F.col("order_value")
                ).otherwise(0)
            ),
            2
        ).alias("revenue")
    )
    .withColumn(
        "on_time_delivery_rate",
        F.round(
            F.col("on_time_orders") /
            F.col("delivered_orders") * 100,
            2
        )
    )
    .orderBy(F.desc("total_orders"))
)

display(gold_category)

categories,total_orders,delivered_orders,cancelled_orders,on_time_orders,late_orders,avg_delivery_days,revenue,on_time_delivery_rate
Toys,386,363,23,255,108,5.83,58498.98,70.25
Grocery,381,361,20,251,110,5.91,60798.97,69.53
Sports,375,357,18,250,107,5.78,56894.68,70.03
Automotive,374,357,17,260,97,5.53,50515.22,72.83
Books,373,356,17,257,99,5.64,50523.19,72.19
Home & Kitchen,371,356,15,249,107,5.74,60614.31,69.94
Electronics,360,344,16,236,108,5.85,51184.49,68.6
Furniture,360,346,14,250,96,5.73,56023.96,72.25
Beauty,356,335,21,232,103,5.87,51569.28,69.25
Fashion,346,329,17,217,112,6.07,45316.91,65.96


In [0]:
silver.select("categories", "unique_products").printSchema()

root
 |-- categories: string (nullable = true)
 |-- unique_products: long (nullable = true)



In [0]:
# GOLD 5 - CANCELLATION INTELLIGENCE
gold_cancellation = (
    silver
    .withColumn(
        "order_month",
        F.date_format("order_date", "yyyy-MM")
    )
    .groupBy("order_month")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(
                F.col("order_status") == "Cancelled", 1
            ).otherwise(0)
        ).alias("cancelled_orders")
    )
    .withColumn(
        "cancellation_rate",
        F.round(
            F.col("cancelled_orders") /
            F.col("total_orders") * 100,
            2
        )
    )
    .orderBy("order_month")
)

display(gold_cancellation)

order_month,total_orders,cancelled_orders,cancellation_rate
2025-01,846,43,5.08
2025-02,792,33,4.17
2025-03,846,40,4.73
2025-04,808,23,2.85
2025-05,825,46,5.58
2025-06,793,41,5.17
2025-07,846,47,5.56
2025-08,891,43,4.83
2025-09,876,48,5.48
2025-10,841,39,4.64


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.ecommerce_2305349.gold;

In [0]:
# SAVE GOLD LAYER
GOLD_BASE = "/Volumes/workspace/ecommerce_2305349/gold"

gold_datasets = {
    "executive_kpis": gold_kpis,
    "monthly_delivery": gold_monthly,
    "seller_performance": gold_sellers,
    "category_performance": gold_category,
    "cancellation_intelligence": gold_cancellation
}

for name, df in gold_datasets.items():
    path = f"{GOLD_BASE}/{name}"

    (
        df.write
        .mode("overwrite")
        .format("delta")
        .save(path)
    )

    print(f"Written: {name} -> {path}")

Written: executive_kpis -> /Volumes/workspace/ecommerce_2305349/gold/executive_kpis
Written: monthly_delivery -> /Volumes/workspace/ecommerce_2305349/gold/monthly_delivery
Written: seller_performance -> /Volumes/workspace/ecommerce_2305349/gold/seller_performance
Written: category_performance -> /Volumes/workspace/ecommerce_2305349/gold/category_performance
Written: cancellation_intelligence -> /Volumes/workspace/ecommerce_2305349/gold/cancellation_intelligence


In [0]:
%sh
databricks secrets list-scopes

The Databricks CLI is only supported for interactive use from the web terminal. We recommend using the web terminal to invoke CLI commands. If you would like to interface with Databricks APIs then we recommend using the Databricks Python SDK.


In [0]:
SNOWFLAKE_OPTIONS = {
    "sfURL": "OVSMJDX-MB70797.snowflakecomputing.com",
    "sfUser": "ABESHAMITRA",
    "sfPassword": dbutils.secrets.get(
        scope="snowflake-project",
        key="password"
    ),
    "sfDatabase": "ECOMMERCE_2305349",
    "sfSchema": "GOLD",
    "sfWarehouse": "ECOMMERCE_WH_2305349",
    "sfRole": "ACCOUNTADMIN"
}

print("Snowflake connection settings loaded securely.")


Snowflake connection settings loaded securely.


In [0]:
test_df = (
    spark.read
    .format("snowflake")
    .options(**SNOWFLAKE_OPTIONS)
    .option(
        "query",
        """
        SELECT
            CURRENT_USER() AS USER_NAME,
            CURRENT_WAREHOUSE() AS WAREHOUSE,
            CURRENT_DATABASE() AS DATABASE_NAME,
            CURRENT_SCHEMA() AS SCHEMA_NAME
        """
    )
    .load()
)

display(test_df)

USER_NAME,WAREHOUSE,DATABASE_NAME,SCHEMA_NAME
ABESHAMITRA,ECOMMERCE_WH_2305349,ECOMMERCE_2305349,GOLD


In [0]:
gold_paths = {
    "EXECUTIVE_KPIS": "/Volumes/workspace/ecommerce_2305349/gold/executive_kpis",
    "MONTHLY_DELIVERY": "/Volumes/workspace/ecommerce_2305349/gold/monthly_delivery",
    "SELLER_PERFORMANCE": "/Volumes/workspace/ecommerce_2305349/gold/seller_performance",
    "CATEGORY_PERFORMANCE": "/Volumes/workspace/ecommerce_2305349/gold/category_performance",
    "CANCELLATION_INTELLIGENCE": "/Volumes/workspace/ecommerce_2305349/gold/cancellation_intelligence"
}

for table_name, path in gold_paths.items():
    df = spark.read.format("delta").load(path)
    
    (
        df.write
        .format("snowflake")
        .options(**SNOWFLAKE_OPTIONS)
        .option("dbtable", table_name)
        .mode("overwrite")
        .save()
    )
    
    print(f"Uploaded: {table_name} | Rows: {df.count()}")

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-7178840619047591>, line 18
      9 for table_name, path in gold_paths.items():
     10     df = spark.read.format("delta").load(path)
     12     (
     13         df.write
     14         .format("snowflake")
     15         .options(**SNOWFLAKE_OPTIONS)
     16         .option("dbtable", table_name)
     17         .mode("overwrite")
---> 18         .save()
     19     )
     21     print(f"Uploaded: {table_name} | Rows: {df.count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._call

In [0]:
from pyspark.sql import DataFrame

gold_vars = {
    name: value
    for name, value in globals().items()
    if isinstance(value, DataFrame)
}

for name, df in gold_vars.items():
    print(f"{name} -> {df.count()} rows | {df.columns}")

test_df -> 1 rows | ['USER_NAME', 'WAREHOUSE', 'DATABASE_NAME', 'SCHEMA_NAME']


In [0]:
SNOWFLAKE_WRITE_OPTIONS = {
    "host": "OVSMJDX-MB70797.snowflakecomputing.com",
    "sfUser": "ABESHAMITRA",
    "sfPassword": dbutils.secrets.get(
        scope="snowflake-project",
        key="password"
    ),
    "sfDatabase": "ECOMMERCE_2305349",
    "sfSchema": "GOLD",
    "sfWarehouse": "ECOMMERCE_WH_2305349",
    "sfRole": "ACCOUNTADMIN"
}

print("Snowflake write settings loaded.")

Snowflake write settings loaded.


In [0]:
gold_paths = {
    "EXECUTIVE_KPIS": "/Volumes/workspace/ecommerce_2305349/gold/executive_kpis",
    "MONTHLY_DELIVERY": "/Volumes/workspace/ecommerce_2305349/gold/monthly_delivery",
    "SELLER_PERFORMANCE": "/Volumes/workspace/ecommerce_2305349/gold/seller_performance",
    "CATEGORY_PERFORMANCE": "/Volumes/workspace/ecommerce_2305349/gold/category_performance",
    "CANCELLATION_INTELLIGENCE": "/Volumes/workspace/ecommerce_2305349/gold/cancellation_intelligence"
}

for table_name, path in gold_paths.items():
    df = spark.read.format("delta").load(path)

    (
        df.write
        .format("snowflake")
        .options(**SNOWFLAKE_WRITE_OPTIONS)
        .option("dbtable", table_name)
        .mode("overwrite")
        .save()
    )

    print(f"Uploaded: {table_name} | Rows: {df.count()}")

Uploaded: EXECUTIVE_KPIS | Rows: 1
Uploaded: MONTHLY_DELIVERY | Rows: 12
Uploaded: SELLER_PERFORMANCE | Rows: 100
Uploaded: CATEGORY_PERFORMANCE | Rows: 175
Uploaded: CANCELLATION_INTELLIGENCE | Rows: 12
